In [1]:
# Land cover #
lc_dnk_23 = "/home/georg/data/LEON_P5_BII/EO_data_prep/LC_with_secondary_veg/lc_sv_dnk_2023.tif"
lc_dnk_21 = "/home/georg/data/LEON_P5_BII/EO_data_prep/LC_with_secondary_veg/lc_sv_dnk_2021.tif"
lc_dnk_18 = "/home/georg/data/LEON_P5_BII/EO_data_prep/LC_with_secondary_veg/lc_sv_dnk_2018.tif"

lc_nld_23 = "/home/georg/data/LEON_P5_BII/EO_data_prep/LC_with_secondary_veg/lc_sv_nld_2023.tif"
lc_nld_21 = "/home/georg/data/LEON_P5_BII/EO_data_prep/LC_with_secondary_veg/lc_sv_nld_2021.tif"
lc_nld_18 = "/home/georg/data/LEON_P5_BII/EO_data_prep/LC_with_secondary_veg/lc_sv_nld_2018.tif"

# Fragmentation #
eca_dnk_23 = "/home/georg/data/LEON_P5_BII/EO_data_prep/ECA/eca_dnk_2023.tif"
eca_dnk_21 = "/home/georg/data/LEON_P5_BII/EO_data_prep/ECA/eca_dnk_2021.tif"
eca_dnk_18 = "/home/georg/data/LEON_P5_BII/EO_data_prep/ECA/eca_dnk_2018.tif"

eca_nld_23 = "/home/georg/data/LEON_P5_BII/EO_data_prep/ECA/eca_nld_2023.tif"
eca_nld_21 = "/home/georg/data/LEON_P5_BII/EO_data_prep/ECA/eca_nld_2021.tif"
eca_nld_18 = "/home/georg/data/LEON_P5_BII/EO_data_prep/ECA/eca_nld_2018.tif"

# Nitrogen #
nit_dnk = "/home/georg/data/LEON_P5_BII/EO_data_prep/Nitrogen/nitrogen_nld_2020.tif"
nit_nld = "/home/georg/data/LEON_P5_BII/EO_data_prep/Nitrogen/nitrogen_nld_2020.tif"

# Bare #
bare_dnk_23 = "/home/georg/data/LEON_P5_BII/EO_data_prep/Bare_Total/bare_total_dnk_2023.tif"
bare_dnk_21 = "/home/georg/data/LEON_P5_BII/EO_data_prep/Bare_Total/bare_total_dnk_2021.tif"
bare_dnk_18 = "/home/georg/data/LEON_P5_BII/EO_data_prep/Bare_Total/bare_total_dnk_2018.tif"

bare_ndl_23 = "/home/georg/data/LEON_P5_BII/EO_data_prep/Bare_Total/bare_total_nld_2023.tif"
bare_ndl_21 = "/home/georg/data/LEON_P5_BII/EO_data_prep/Bare_Total/bare_total_nld_2021.tif"
bare_ndl_18 = "/home/georg/data/LEON_P5_BII/EO_data_prep/Bare_Total/bare_total_nld_2018.tif"

# Settlement #
ghsl_dnk_23 = "/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_dnk_2025.tif"
ghsl_dnk_21 = "/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_dnk_2020.tif"
ghsl_dnk_18 = "/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_dnk_2015.tif"

ghsl_nld_23 = "/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_nld_2025.tif"
ghsl_nld_21 = "/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_nld_2020.tif"
ghsl_nld_18 = "/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_nld_2015.tif"

# # Classes
# lc_classes = {
#     1: 'Sealed',
#     21: 'SV Mature',
#     22: 'SV Intermediate',
#     23: 'SV Young',
#     24: 'SV Indeterminate',
#     3: 'Plantation',
#     6: 'Permanent herbaceous',
#     7: 'Periodically herbaceous',
#     8: 'Lichens and mosses',
#     9: 'Non and sparsely vegetated',
#     10: 'Water',
#     11: 'Snow and ice',
#     253: 'Coastal sea buffer'
#     254: 'Outside area'
#     255: 'No data'
# }

## LU Intensity based on Fragmentation

In [ ]:
import rasterio
import numpy as np
from rasterio.windows import Window
import gc

# ============================================
# Thresholds - ADJUST HERE
# ============================================
THRESHOLDS = {
    'sv': {'low': 33, 'high': 66},
    'plantation': {'low': 33, 'high': 66},
    'crop': {'low': 50, 'high': 100},
    'pasture': {'low': 450, 'high': 650},
    'settlement': {'low': 20, 'high': 80},
}

# ============================================
# Process Country Year
# ============================================
year, country = 2018, 'dnk'
lc_file, eca_file, bare_file, ghsl_file, nit_file = lc_dnk_18, eca_dnk_18, bare_dnk_18, ghsl_dnk_18, nit_dnk

print(f"\n{'='*60}")
print(f"Processing {country.upper()} {year}")
print('='*60)

# Load auxiliary layers
print("\nLoading auxiliary layers...")
with rasterio.open(eca_file) as src:
    eca = src.read(1)
with rasterio.open(bare_file) as src:
    bare = src.read(1)
with rasterio.open(ghsl_file) as src:
    ghsl = src.read(1)
with rasterio.open(nit_file) as src:
    nitrogen = src.read(1)

# ============================================
# CHECK AUXILIARY LAYER DISTRIBUTIONS
# ============================================
print(f"\nAUXILIARY LAYER DISTRIBUTIONS (using current thresholds):")

# ECA (for SV & Plantation)
t = THRESHOLDS['sv']
print(f"\n  ECA (for SV & Plantation):")
print(f"  Min: {eca.min():.1f}, Max: {eca.max():.1f}, Mean: {eca.mean():.1f}")
print(f"  <{t['low']} (intense): {np.sum(eca < t['low']) / eca.size * 100:.1f}%")
print(f"  {t['low']}-{t['high']} (light): {np.sum((eca >= t['low']) & (eca <= t['high'])) / eca.size * 100:.1f}%")
print(f"  >{t['high']} (minimal): {np.sum(eca > t['high']) / eca.size * 100:.1f}%")

# Bare (for Crop)
t = THRESHOLDS['crop']
print(f"\n  Bare (for Crop):")
print(f"  Min: {bare.min():.1f}, Max: {bare.max():.1f}, Mean: {bare.mean():.1f}")
print(f"  <{t['low']} (minimal): {np.sum(bare < t['low']) / bare.size * 100:.1f}%")
print(f"  {t['low']}-{t['high']} (light): {np.sum((bare >= t['low']) & (bare <= t['high'])) / bare.size * 100:.1f}%")
print(f"  >{t['high']} (intense): {np.sum(bare > t['high']) / bare.size * 100:.1f}%")

# GHSL (for Settlement)
t = THRESHOLDS['settlement']
print(f"\n  GHSL (for Settlement):")
print(f"  Min: {ghsl.min():.1f}, Max: {ghsl.max():.1f}, Mean: {ghsl.mean():.1f}")
print(f"  Valid (>=0): {np.sum(ghsl >= 0) / ghsl.size * 100:.1f}%")
print(f"  <{t['low']} (minimal): {np.sum((ghsl >= 0) & (ghsl < t['low'])) / ghsl.size * 100:.1f}%")
print(f"  {t['low']}-{t['high']} (light): {np.sum((ghsl >= t['low']) & (ghsl <= t['high'])) / ghsl.size * 100:.1f}%")
print(f"  >{t['high']} (intense): {np.sum(ghsl > t['high']) / ghsl.size * 100:.1f}%")

# Nitrogen (for Pasture)
t = THRESHOLDS['pasture']
print(f"\n  Nitrogen (for Pasture):")
print(f"  Min: {nitrogen.min():.1f}, Max: {nitrogen.max():.1f}, Mean: {nitrogen.mean():.1f}")
print(f"  <{t['low']} (minimal): {np.sum(nitrogen < t['low']) / nitrogen.size * 100:.1f}%")
print(f"  {t['low']}-{t['high']} (light): {np.sum((nitrogen >= t['low']) & (nitrogen <= t['high'])) / nitrogen.size * 100:.1f}%")
print(f"  >{t['high']} (intense): {np.sum(nitrogen > t['high']) / nitrogen.size * 100:.1f}%")

# ============================================
# Process LC in chunks
# ============================================
print(f"\n{'='*60}")
print("Starting classification...")

with rasterio.open(lc_file) as lc_src:
    profile = lc_src.profile
    height, width = lc_src.shape
    
    # Calculate scale factors
    eca_scale_row = height / eca.shape[0]
    eca_scale_col = width / eca.shape[1]
    bare_scale_row = height / bare.shape[0]
    bare_scale_col = width / bare.shape[1]
    ghsl_scale_row = height / ghsl.shape[0]
    ghsl_scale_col = width / ghsl.shape[1]
    nit_scale_row = height / nitrogen.shape[0]
    nit_scale_col = width / nitrogen.shape[1]
    
    print(f"\nResolution scale factors:")
    print(f"  ECA: {eca_scale_row:.1f}x, Bare: {bare_scale_row:.1f}x, GHSL: {ghsl_scale_row:.1f}x, Nitrogen: {nit_scale_row:.1f}x")
    
    # Create output array
    intensity = np.zeros((height, width), dtype=profile['dtype'])
    
    CHUNK_SIZE = 4000
    
    for row_start in range(0, height, CHUNK_SIZE):
        row_end = min(row_start + CHUNK_SIZE, height)
        print(f"  Processing rows {row_start}-{row_end}/{height}...")
        
        window = Window(0, row_start, width, row_end - row_start)
        lc = lc_src.read(1, window=window)
        chunk_intensity = lc.copy()
        
        # --- SV ---
        sv_mask = np.isin(lc, [21, 22, 23, 24])
        sv_rows, sv_cols = np.where(sv_mask)
        if len(sv_rows) > 0:
            global_rows = sv_rows + row_start
            eca_rows = np.clip((global_rows / eca_scale_row).astype(int), 0, eca.shape[0] - 1)
            eca_cols = np.clip((sv_cols / eca_scale_col).astype(int), 0, eca.shape[1] - 1)
            eca_values = eca[eca_rows, eca_cols]
            sv_classes = lc[sv_rows, sv_cols]
            t = THRESHOLDS['sv']
            intensity_class = np.where(eca_values < t['low'], 0,
                                       np.where(eca_values <= t['high'], 1, 2))
            chunk_intensity[sv_rows, sv_cols] = sv_classes * 10 + intensity_class
        
        # --- Plantation ---
        pl_mask = (lc == 3)
        pl_rows, pl_cols = np.where(pl_mask)
        if len(pl_rows) > 0:
            global_rows = pl_rows + row_start
            eca_rows = np.clip((global_rows / eca_scale_row).astype(int), 0, eca.shape[0] - 1)
            eca_cols = np.clip((pl_cols / eca_scale_col).astype(int), 0, eca.shape[1] - 1)
            eca_values = eca[eca_rows, eca_cols]
            t = THRESHOLDS['plantation']
            intensity_class = np.where(eca_values < t['low'], 0,
                                       np.where(eca_values <= t['high'], 1, 2))
            chunk_intensity[pl_rows, pl_cols] = 30 + intensity_class
        
        # --- Crop ---
        crop_mask = (lc == 7)
        crop_rows, crop_cols = np.where(crop_mask)
        if len(crop_rows) > 0:
            global_rows = crop_rows + row_start
            bare_rows = np.clip((global_rows / bare_scale_row).astype(int), 0, bare.shape[0] - 1)
            bare_cols = np.clip((crop_cols / bare_scale_col).astype(int), 0, bare.shape[1] - 1)
            bare_values = bare[bare_rows, bare_cols]
            t = THRESHOLDS['crop']
            intensity_class = np.where(bare_values < t['low'], 0,
                                       np.where(bare_values <= t['high'], 1, 2))
            chunk_intensity[crop_rows, crop_cols] = 70 + intensity_class
        
        # --- Pasture ---
        pasture_mask = (lc == 6)
        pasture_rows, pasture_cols = np.where(pasture_mask)
        if len(pasture_rows) > 0:
            nit_rows = np.clip((pasture_rows / nit_scale_row).astype(int), 0, nitrogen.shape[0] - 1)
            nit_cols = np.clip((pasture_cols / nit_scale_col).astype(int), 0, nitrogen.shape[1] - 1)
            nit_values = nitrogen[nit_rows, nit_cols]
            t = THRESHOLDS['pasture']
            intensity_class = np.where(nit_values < t['low'], 0,
                                       np.where(nit_values <= t['high'], 1, 2))
            chunk_intensity[pasture_rows, pasture_cols] = 60 + intensity_class
        
        # --- Settlement ---
        settlement_mask = (lc == 1)
        settlement_rows, settlement_cols = np.where(settlement_mask)
        if len(settlement_rows) > 0:
            global_rows = settlement_rows + row_start
            ghsl_rows = np.clip((global_rows / ghsl_scale_row).astype(int), 0, ghsl.shape[0] - 1)
            ghsl_cols = np.clip((settlement_cols / ghsl_scale_col).astype(int), 0, ghsl.shape[1] - 1)
            ghsl_values = ghsl[ghsl_rows, ghsl_cols]
            t = THRESHOLDS['settlement']
            intensity_class = np.where(ghsl_values < t['low'], 0,
                                       np.where(ghsl_values <= t['high'], 1, 2))
            chunk_intensity[settlement_rows, settlement_cols] = 10 + intensity_class
        
        # Store chunk
        intensity[row_start:row_end, :] = chunk_intensity
        
        # Clean up chunk
        del lc, chunk_intensity
        gc.collect()

# Store result
intensity_results = {
    f"{country}_{year}": {
        'data': intensity,
        'profile': profile
    }
}

# ============================================
# Statistics
# ============================================
print(f"\n{'='*60}")
print(f"Land Use Intensity Statistics for {country.upper()} {year}:")
print(f"Current thresholds: SV/Plantation ECA<{THRESHOLDS['sv']['low']} (intense), Crop Bare<{THRESHOLDS['crop']['low']} (minimal),")
print(f"                    Pasture N<{THRESHOLDS['pasture']['low']} (minimal), Settlement GHSL<{THRESHOLDS['settlement']['low']} (minimal)")
print(f"\n{'Class':<35} {'Pixels':>15} {'%':>8}")
print(f"{'-'*35} {'-'*15} {'-'*8}")

total_pixels = intensity.size

class_stats = {
    10: 'Settlement minimal', 11: 'Settlement light', 12: 'Settlement intense',
    30: 'Plantation minimal', 31: 'Plantation light', 32: 'Plantation intense',
    60: 'Pasture minimal', 61: 'Pasture light', 62: 'Pasture intense',
    70: 'Crop minimal', 71: 'Crop light', 72: 'Crop intense',
    210: 'SV Mature minimal', 211: 'SV Mature light', 212: 'SV Mature intense',
    220: 'SV Intermediate minimal', 221: 'SV Intermediate light', 222: 'SV Intermediate intense',
    230: 'SV Young minimal', 231: 'SV Young light', 232: 'SV Young intense',
    240: 'SV Indeterminate minimal', 241: 'SV Indeterminate light', 242: 'SV Indeterminate intense',
}

for class_val, class_name in class_stats.items():
    count = np.sum(intensity == class_val)
    proportion = count / total_pixels * 100
    if count > 0:
        print(f"{class_name:<35} {count:>15,} {proportion:>7.3f}%")

# Summary
total_classified = np.sum(np.isin(intensity, list(class_stats.keys())))
print(f"\n{'='*60}")
print(f"Total classified: {total_classified:,} ({total_classified/total_pixels*100:.2f}%)")
print(f"Other/unclassified: {total_pixels - total_classified:,} ({(total_pixels - total_classified)/total_pixels*100:.2f}%)")

# Cleanup
del eca, bare, ghsl, nitrogen
gc.collect()

print(f"\n✅ Done! Results in 'intensity_results' dict")


Processing DNK 2018

Loading auxiliary layers...

AUXILIARY LAYER DISTRIBUTIONS (using current thresholds):

  ECA (for SV & Plantation):
  Min: 0.0, Max: 87.9, Mean: 1.4
  <33 (intense): 99.3%
  33-66 (light): 0.7%
  >66 (minimal): 0.0%

  Bare (for Crop):
  Min: 0.0, Max: 324.0, Mean: 12.4
  <50 (minimal): 91.6%
  50-100 (light): 1.6%
  >100 (intense): 6.8%

  GHSL (for Settlement):
  Min: -200.0, Max: 479.6, Mean: -54.0
  Valid (>=0): 72.6%
  <20 (minimal): 72.0%
  20-80 (light): 0.5%
  >80 (intense): 0.0%

  Nitrogen (for Pasture):
  Min: 0.0, Max: 1222.0, Mean: 307.6
  <450 (minimal): 58.0%
  450-650 (light): 35.1%
  >650 (intense): 6.9%

Starting classification...

Resolution scale factors:
  ECA: 100.2x, Bare: 1.0x, GHSL: 9.3x, Nitrogen: 1.1x
  Processing rows 0-2000/35363...
  Processing rows 2000-4000/35363...
  Processing rows 4000-6000/35363...
  Processing rows 6000-8000/35363...
  Processing rows 8000-10000/35363...
  Processing rows 10000-12000/35363...
  Processing rows

In [ ]:
from pathlib import Path

# Choose what to save
save_country = 'dnk'  # 'dnk' or 'nld'
save_year = 2023      # 2018, 2021, 2023

output_dir = Path("~/data/LEON_P5_BII/EO_data_prep/output").expanduser()
output_dir.mkdir(parents=True, exist_ok=True)

# Save 
key = f"{save_country}_{save_year}"
if key in intensity_results:
    output_file = output_dir / f"lc_intensity_{save_country}_{save_year}.tif"
    
    with rasterio.open(output_file, 'w', **intensity_results[key]['profile']) as dst:
        dst.write(intensity_results[key]['data'], 1)
    
    print(f"✅ Saved: {output_file}")
else:
    print(f"❌ {key} not found in results")

## All together

In [5]:
import rasterio
import numpy as np
from rasterio.windows import Window
from pathlib import Path


# ============================================
# Thresholds
# ============================================
THRESHOLDS = {
    'sv': {'low': 33, 'high': 66},
    'plantation': {'low': 33, 'high': 66},
    'crop': {'low': 50, 'high': 100},
    'pasture': {'low': 450, 'high': 650},
    'settlement': {'low': 20, 'high': 80},
}

# ============================================
# Files
# ============================================
files = [
    (2018, 'dnk', lc_dnk_18, eca_dnk_18, bare_dnk_18, ghsl_dnk_18, nit_dnk),
    (2021, 'dnk', lc_dnk_21, eca_dnk_21, bare_dnk_21, ghsl_dnk_21, nit_dnk),
    (2023, 'dnk', lc_dnk_23, eca_dnk_23, bare_dnk_23, ghsl_dnk_23, nit_dnk),
    (2018, 'nld', lc_nld_18, eca_nld_18, bare_ndl_18, ghsl_nld_18, nit_nld),
    (2021, 'nld', lc_nld_21, eca_nld_21, bare_ndl_21, ghsl_nld_21, nit_nld),
    (2023, 'nld', lc_nld_23, eca_nld_23, bare_ndl_23, ghsl_nld_23, nit_nld),
]

CHUNK_SIZE = 4000

for year, country, lc_file, eca_file, bare_file, ghsl_file, nit_file in files:
    print(f"\nProcessing {country}_{year}...")

    # Create output directory
    output_dir = Path(f"~/data/LEON_P5_BII/BII_LU_layer/Land_use_map").expanduser()
    output_dir.mkdir(parents=True, exist_ok=True)    
    
    # Load auxiliary layers
    with rasterio.open(eca_file) as src:
        eca = src.read(1)
    with rasterio.open(bare_file) as src:
        bare = src.read(1)
    with rasterio.open(ghsl_file) as src:
        ghsl = src.read(1)
    with rasterio.open(nit_file) as src:
        nitrogen = src.read(1)
    
    # Process LC in chunks
    with rasterio.open(lc_file) as lc_src:
        profile = lc_src.profile
        height, width = lc_src.shape
        
        # Calculate scale factors
        eca_scale_row = height / eca.shape[0]
        eca_scale_col = width / eca.shape[1]
        bare_scale_row = height / bare.shape[0]
        bare_scale_col = width / bare.shape[1]
        ghsl_scale_row = height / ghsl.shape[0]
        ghsl_scale_col = width / ghsl.shape[1]
        nit_scale_row = height / nitrogen.shape[0]
        nit_scale_col = width / nitrogen.shape[1]
        
        print(f"  Scale factors: ECA={eca_scale_row:.1f}x, Bare={bare_scale_row:.1f}x, GHSL={ghsl_scale_row:.1f}x, Nitrogen={nit_scale_row:.1f}x")
        
        output_file = output_dir / f"LU_map_{country}_{year}.tif"
        
        with rasterio.open(output_file, 'w', **profile) as dst:
            
            for row_start in range(0, height, CHUNK_SIZE):
                row_end = min(row_start + CHUNK_SIZE, height)
                print(f"  Rows {row_start}-{row_end}/{height}")
                
                window = Window(0, row_start, width, row_end - row_start)
                lc = lc_src.read(1, window=window)
                intensity = lc.copy()
                
                # --- SV ---
                sv_mask = np.isin(lc, [21, 22, 23, 24])
                sv_rows, sv_cols = np.where(sv_mask)
                if len(sv_rows) > 0:
                    global_rows = sv_rows + row_start
                    eca_rows = np.clip((global_rows / eca_scale_row).astype(int), 0, eca.shape[0] - 1)
                    eca_cols = np.clip((sv_cols / eca_scale_col).astype(int), 0, eca.shape[1] - 1)
                    eca_values = eca[eca_rows, eca_cols]
                    sv_classes = lc[sv_rows, sv_cols]
                    t = THRESHOLDS['sv']
                    intensity_class = np.where(eca_values < t['low'], 0,
                                               np.where(eca_values <= t['high'], 1, 2))
                    intensity[sv_rows, sv_cols] = sv_classes * 10 + intensity_class
                
                # --- Plantation ---
                pl_mask = (lc == 3)
                pl_rows, pl_cols = np.where(pl_mask)
                if len(pl_rows) > 0:
                    global_rows = pl_rows + row_start
                    eca_rows = np.clip((global_rows / eca_scale_row).astype(int), 0, eca.shape[0] - 1)
                    eca_cols = np.clip((pl_cols / eca_scale_col).astype(int), 0, eca.shape[1] - 1)
                    eca_values = eca[eca_rows, eca_cols]
                    t = THRESHOLDS['plantation']
                    intensity_class = np.where(eca_values < t['low'], 0,
                                               np.where(eca_values <= t['high'], 1, 2))
                    intensity[pl_rows, pl_cols] = 30 + intensity_class
                
                # --- Crop ---
                crop_mask = (lc == 7)
                crop_rows, crop_cols = np.where(crop_mask)
                if len(crop_rows) > 0:
                    global_rows = crop_rows + row_start
                    bare_rows = np.clip((global_rows / bare_scale_row).astype(int), 0, bare.shape[0] - 1)
                    bare_cols = np.clip((crop_cols / bare_scale_col).astype(int), 0, bare.shape[1] - 1)
                    bare_values = bare[bare_rows, bare_cols]
                    t = THRESHOLDS['crop']
                    intensity_class = np.where(bare_values < t['low'], 0,
                                               np.where(bare_values <= t['high'], 1, 2))
                    intensity[crop_rows, crop_cols] = 70 + intensity_class
                
                # --- Pasture ---
                pasture_mask = (lc == 6)
                pasture_rows, pasture_cols = np.where(pasture_mask)
                if len(pasture_rows) > 0:
                    nit_rows = np.clip((pasture_rows / nit_scale_row).astype(int), 0, nitrogen.shape[0] - 1)
                    nit_cols = np.clip((pasture_cols / nit_scale_col).astype(int), 0, nitrogen.shape[1] - 1)
                    nit_values = nitrogen[nit_rows, nit_cols]
                    t = THRESHOLDS['pasture']
                    intensity_class = np.where(nit_values < t['low'], 0,
                                               np.where(nit_values <= t['high'], 1, 2))
                    intensity[pasture_rows, pasture_cols] = 60 + intensity_class
                
                # --- Settlement ---
                settlement_mask = (lc == 1)
                settlement_rows, settlement_cols = np.where(settlement_mask)
                if len(settlement_rows) > 0:
                    global_rows = settlement_rows + row_start
                    ghsl_rows = np.clip((global_rows / ghsl_scale_row).astype(int), 0, ghsl.shape[0] - 1)
                    ghsl_cols = np.clip((settlement_cols / ghsl_scale_col).astype(int), 0, ghsl.shape[1] - 1)
                    ghsl_values = ghsl[ghsl_rows, ghsl_cols]
                    t = THRESHOLDS['settlement']
                    intensity_class = np.where(ghsl_values < t['low'], 0,
                                               np.where(ghsl_values <= t['high'], 1, 2))
                    intensity[settlement_rows, settlement_cols] = 10 + intensity_class
                
                # Write chunk
                dst.write(intensity, 1, window=window)
    
    print(f"✅ Saved: {output_file}")

print("\n🎯 All done!")


Processing dnk_2018...
  Scale factors: ECA=100.2x, Bare=1.0x, GHSL=9.3x, Nitrogen=1.1x
  Rows 0-4000/35363
  Rows 4000-8000/35363
  Rows 8000-12000/35363
  Rows 12000-16000/35363
  Rows 16000-20000/35363
  Rows 20000-24000/35363
  Rows 24000-28000/35363
  Rows 28000-32000/35363
  Rows 32000-35363/35363
✅ Saved: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_map/LU_map_dnk_2018.tif

Processing dnk_2021...
  Scale factors: ECA=100.2x, Bare=1.0x, GHSL=9.3x, Nitrogen=1.1x
  Rows 0-4000/35363
  Rows 4000-8000/35363
  Rows 8000-12000/35363
  Rows 12000-16000/35363
  Rows 16000-20000/35363
  Rows 20000-24000/35363
  Rows 24000-28000/35363
  Rows 28000-32000/35363
  Rows 32000-35363/35363
✅ Saved: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_map/LU_map_dnk_2021.tif

Processing dnk_2023...
  Scale factors: ECA=100.2x, Bare=1.0x, GHSL=9.3x, Nitrogen=1.1x
  Rows 0-4000/35363
  Rows 4000-8000/35363
  Rows 8000-12000/35363
  Rows 12000-16000/35363
  Rows 16000-20000/35363
  Rows 20000-2

## Create 1km Grid cell with proportions

In [6]:
import rasterio
import numpy as np
from pathlib import Path

# ============================================
# Define intensity classes
# ============================================
INTENSITY_CLASSES = {
    'settlement_minimal': 10,
    'settlement_light': 11,
    'settlement_intense': 12,
    'plantation_minimal': 30,
    'plantation_light': 31,
    'plantation_intense': 32,
    'pasture_minimal': 60,
    'pasture_light': 61,
    'pasture_intense': 62,
    'crop_minimal': 70,
    'crop_light': 71,
    'crop_intense': 72,
    'sv_mature_minimal': 210,
    'sv_mature_light': 211,
    'sv_mature_intense': 212,
    'sv_intermediate_minimal': 220,
    'sv_intermediate_light': 221,
    'sv_intermediate_intense': 222,
    'sv_young_minimal': 230,
    'sv_young_light': 231,
    'sv_young_intense': 232,
    'sv_indeterminate_minimal': 240,
    'sv_indeterminate_light': 241,
    'sv_indeterminate_intense': 242,
}

# ============================================
# Choose what to process
# ============================================
country = 'dnk'  # 'dnk' or 'nld'
year = 2023      # 2018, 2021, or 2023

# ============================================
# Calculate proportions
# ============================================
print(f"Processing {country.upper()} {year}...")

# Load from new directory structure
input_file = Path(f"~/data/LEON_P5_BII/BII_LU_layer/Land_use_map/LU_map_{country}_{year}.tif").expanduser()
print(f"Loading: {input_file}")


with rasterio.open(input_file) as src:
    intensity = src.read(1)
    height, width = src.shape
    
    # 1km grid dimensions
    grid_height = height // 100
    grid_width = width // 100
    
    print(f"Original: {height} x {width} (10m)")
    print(f"Grid: {grid_height} x {grid_width} (1km)")
    
    # Store profile for later
    transform = src.transform * src.transform.scale(100, 100)
    output_profile = src.profile.copy()
    output_profile.update({
        'height': grid_height,
        'width': grid_width,
        'transform': transform,
        'dtype': 'float32'
    })
    
    # Calculate proportions for all classes
    proportion_grids = {}
    
    for class_name, class_value in INTENSITY_CLASSES.items():
        print(f"  Calculating {class_name}...")
        
        grid = np.zeros((grid_height, grid_width), dtype=np.float32)
        
        # Aggregate to 1km
        for i in range(grid_height):
            for j in range(grid_width):
                row_start = i * 100
                col_start = j * 100
                row_end = min(row_start + 100, height)
                col_end = min(col_start + 100, width)
                
                block = intensity[row_start:row_end, col_start:col_end]
                
                total_pixels = block.size
                class_pixels = np.sum(block == class_value)
                grid[i, j] = class_pixels / total_pixels if total_pixels > 0 else 0
        
        proportion_grids[class_name] = grid
        
        # Quick stats
        non_zero = np.sum(grid > 0)
        if non_zero > 0:
            print(f"    ✓ {non_zero} cells with this class")

print(f"\n✅ Proportions calculated and stored in 'proportion_grids' dict")

Processing DNK 2023...
Loading: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_map/LU_map_dnk_2023.tif
Original: 35363 x 45095 (10m)
Grid: 353 x 450 (1km)
  Calculating settlement_minimal...
    ✓ 44935 cells with this class
  Calculating settlement_light...
    ✓ 1748 cells with this class
  Calculating settlement_intense...
    ✓ 178 cells with this class
  Calculating plantation_minimal...
    ✓ 44926 cells with this class
  Calculating plantation_light...
    ✓ 2957 cells with this class
  Calculating plantation_intense...
    ✓ 149 cells with this class
  Calculating pasture_minimal...
    ✓ 45487 cells with this class
  Calculating pasture_light...
    ✓ 1847 cells with this class
  Calculating pasture_intense...
    ✓ 608 cells with this class
  Calculating crop_minimal...
    ✓ 42753 cells with this class
  Calculating crop_light...
    ✓ 23380 cells with this class
  Calculating crop_intense...
    ✓ 36203 cells with this class
  Calculating sv_mature_minimal...
    ✓ 2691

In [ ]:
# ============================================
# Save ALL classes
# ============================================
classes_to_save = list(INTENSITY_CLASSES.keys())  # All 24 classes

# Create organized output directory
output_dir = Path(f"~/data/LEON_P5_BII/BII_LU_layer/Land_use_proportions/LU_prop_{country}_{year}").expanduser()
output_dir.mkdir(parents=True, exist_ok=True)

print(f"\nSaving all {len(classes_to_save)} layers for {country.upper()} {year}...")
print(f"Output directory: {output_dir}")

for class_name in classes_to_save:
    if class_name in proportion_grids:
        output_file = output_dir / f"{class_name}.tif"
        
        with rasterio.open(output_file, 'w', **output_profile) as dst:
            dst.write(proportion_grids[class_name], 1)
        
        # Show stats for non-empty layers
        non_zero = np.sum(proportion_grids[class_name] > 0)
        if non_zero > 0:
            max_prop = np.max(proportion_grids[class_name])
            print(f"  ✓ {class_name:<30} ({non_zero:>5} cells, max={max_prop:.3f})")
        else:
            print(f"  ○ {class_name:<30} (empty)")
    else:
        print(f"  ✗ {class_name} not found")

print(f"\n🎯 Done! Saved 24 layers to: {output_dir}/")


Saving all 24 layers for DNK 2023...
Output directory: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_proportions/LU_prop_dnk_2023
  ✓ settlement_minimal             (44935 cells, max=1.000)
  ✓ settlement_light               ( 1748 cells, max=0.805)
  ✓ settlement_intense             (  178 cells, max=0.848)
  ✓ plantation_minimal             (44926 cells, max=0.996)
  ✓ plantation_light               ( 2957 cells, max=0.611)
  ✓ plantation_intense             (  149 cells, max=0.277)
  ✓ pasture_minimal                (45487 cells, max=1.000)
  ✓ pasture_light                  ( 1847 cells, max=0.929)
  ✓ pasture_intense                (  608 cells, max=0.991)
  ✓ crop_minimal                   (42753 cells, max=0.998)
  ✓ crop_light                     (23380 cells, max=0.816)
  ✓ crop_intense                   (36203 cells, max=0.970)
  ✓ sv_mature_minimal              (26918 cells, max=0.572)
  ✓ sv_mature_light                ( 2537 cells, max=0.541)
  ✓ sv_mature_intense   

In [9]:
# ============================================
# Save as multi-band GeoTIFF 
# ============================================
output_dir = Path(f"~/data/LEON_P5_BII/BII_LU_layer/Land_use_proportions").expanduser()
output_file = output_dir / f"LU_prop_{country}_{year}.tif"

# Update profile for multi-band
profile_multiband = output_profile.copy()
profile_multiband.update({'count': len(INTENSITY_CLASSES)})

print(f"\nSaving multi-band GeoTIFF for {country.upper()} {year}...")
with rasterio.open(output_file, 'w', **profile_multiband) as dst:
    for i, (class_name, grid) in enumerate(proportion_grids.items(), 1):
        dst.write(grid, i)
        dst.set_band_description(i, class_name)
        print(f"  Band {i:2d}: {class_name}")

print(f"✅ Saved: {output_file}")


Saving multi-band GeoTIFF for DNK 2023...
  Band  1: settlement_minimal
  Band  2: settlement_light
  Band  3: settlement_intense
  Band  4: plantation_minimal
  Band  5: plantation_light
  Band  6: plantation_intense
  Band  7: pasture_minimal
  Band  8: pasture_light
  Band  9: pasture_intense
  Band 10: crop_minimal
  Band 11: crop_light
  Band 12: crop_intense
  Band 13: sv_mature_minimal
  Band 14: sv_mature_light
  Band 15: sv_mature_intense
  Band 16: sv_intermediate_minimal
  Band 17: sv_intermediate_light
  Band 18: sv_intermediate_intense
  Band 19: sv_young_minimal
  Band 20: sv_young_light
  Band 21: sv_young_intense
  Band 22: sv_indeterminate_minimal
  Band 23: sv_indeterminate_light
  Band 24: sv_indeterminate_intense
✅ Saved: /home/georg/data/LEON_P5_BII/BII_LU_layer/Land_use_proportions/LU_prop_dnk_2023.tif
